# LoRA Fine-tuning Example

This notebook loads OPT-350M, attaches LoRA adapters, preprocesses Alpaca data, and trains with a small validation split.

In [1]:
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    default_data_collator,
)
from peft import LoraConfig, get_peft_model
from datasets import load_dataset
import torch

c:\Users\SiddheshLihe\LLM_Finetune\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [19]:
import transformers
print('transformers version:', transformers.__version__)
print('TrainingArguments supported kwargs:', 'evaluation_strategy' in transformers.TrainingArguments.__init__.__code__.co_varnames)


transformers version: 5.8.0
TrainingArguments supported kwargs: False


In [20]:
import transformers
print('transformers version:', transformers.__version__)
print('TrainingArguments supported kwargs:', 'evaluation_strategy' in transformers.TrainingArguments.__init__.__code__.co_varnames)

transformers version: 5.8.0
TrainingArguments supported kwargs: False


In [21]:
model_name = "../opt-350m"

device = "cuda" if torch.cuda.is_available() else "cpu"

# Load tokenizer aligned with the model
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id
    tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto" if device == "cuda" else None,
)
model.config.use_cache = False

print(f"Using device: {device}")
print(f"Loaded model: {model_name}")

Loading weights: 100%|██████████| 388/388 [00:00<00:00, 1093.70it/s]


Using device: cuda
Loaded model: ../opt-350m


In [22]:
# Add LoRA adapters to the model
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 786,432 || all params: 331,982,848 || trainable%: 0.2369


In [23]:
# Load Alpaca and create a validation split if needed
raw_dataset = load_dataset("tatsu-lab/alpaca")
if "test" not in raw_dataset:
    raw_dataset = raw_dataset["train"].train_test_split(test_size=0.05, seed=42)

raw_dataset

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output', 'text'],
        num_rows: 49401
    })
    test: Dataset({
        features: ['instruction', 'input', 'output', 'text'],
        num_rows: 2601
    })
})

In [24]:
def format_example(example):
    prompt = "### Instruction:\n" + example["instruction"].strip() + "\n"
    if example.get("input"):
        user_input = example["input"].strip()
        if len(user_input) > 0:
            prompt += "\n### Input:\n" + user_input + "\n"
    prompt += "\n### Response:\n" + example["output"].strip()
    return {"text": prompt}

formatted_dataset = raw_dataset.map(format_example, batched=False)
formatted_dataset

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output', 'text'],
        num_rows: 49401
    })
    test: Dataset({
        features: ['instruction', 'input', 'output', 'text'],
        num_rows: 2601
    })
})

In [25]:
def tokenize(batch):
    outputs = tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=512,
    )
    outputs["labels"] = [ids.copy() for ids in outputs["input_ids"]]
    return outputs

tokenized = formatted_dataset.map(
    tokenize,
    batched=True,
    remove_columns=formatted_dataset["train"].column_names,
)

# Make datasets PyTorch friendly
tokenized.set_format(type="torch")
tokenized

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 49401
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 2601
    })
})

In [26]:
training_args = TrainingArguments(
    output_dir="./opt350m_lora",
    per_device_train_batch_size=8,
    num_train_epochs=3,
    learning_rate=2e-4,
    fp16=(device == "cuda"),
    logging_steps=10,
    save_steps=500,
    save_total_limit=2,
    logging_dir="./logs",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    data_collator=default_data_collator,
)

trainer.train()
trainer.save_model("./opt350m_lora/final")

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Step,Training Loss
10,8.572442
20,2.066683
30,0.581603
40,0.472458
50,0.469719
60,0.417006
70,0.443709
80,0.381024
90,0.383331
100,0.404525


In [27]:
# Optional: inspect a sample prediction after training
input_text = "### Instruction:\nWrite a short poem about machine learning.\n\n### Response:\n"
inputs = tokenizer(input_text, return_tensors="pt").to(model.device)
outputs = model.generate(
    **inputs,
    max_new_tokens=100,
    do_sample=True,
    top_p=0.9,
    temperature=0.8,
)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

### Instruction:
Write a short poem about machine learning.

### Response:
The machines we know are powerful, and we are able to learn and act on them with the help of them.
We can apply the power of machines to solve a complex problem, and we can use them to help our families and communities. 
The machines we use are our friends, and we can use them to make our lives easier, and make our communities better. 

We can use the power of machines to create more efficient and effective businesses, and we can use them to


In [2]:
#model evaluation

In [3]:
def evaluate_model(model,input_text):
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=True,
        top_p=0.9,
        temperature=0.8,
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [6]:
# device = "cuda" if torch.cuda.is_available() else "cpu"
device = "cpu"
model = AutoModelForCausalLM.from_pretrained("./opt350m_lora/final").to(device)
tokenizer = AutoTokenizer.from_pretrained("../opt-350m", use_fast=True)


Loading weights: 100%|██████████| 96/96 [00:00<00:00, 12338.83it/s]


In [9]:
input_text = "### Instruction:\nWrite a 1 line description on low protein density.\n\n### Response:\n"
outputs = evaluate_model(model, input_text)
print(outputs)

### Instruction:
Write a 1 line description on low protein density.

### Response:
The low protein density of low protein sources such as dairy, eggs, and meats is due to a number of factors. First, low protein sources can lead to increased amounts of food allergies and intolerance. Second, low protein sources can lead to higher levels of food allergies and intolerance. Third, low protein sources can lead to increased amounts of energy expenditure and poor energy performance. Finally, low protein sources may lead to decreased levels of immune system function.


In [1]:
134/140*100

95.71428571428572